In [4]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from dotenv import load_dotenv
from langchain_core.documents import Document

In [5]:
load_dotenv()

embd_model = HuggingFaceEmbeddings(model = 'sentence-transformers/all-MiniLM-L6-v2')

# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7830.16it/s]


In [6]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [ ]:
vector_store = Chroma(
    embedding_function = embd_model,
    persist_directory = 'my_chroma_db', #creates a directory my_Chroma_db to store all in hard_disk, if not given as argument, stores in RAM only for current session
    collection_name = 'sample'
)

#using vs = Chroma() starts out with a blank folder, or directs towards an existing folder if mentioned. 
# Using Chroma.from_documents() create brand new space and read all the docs you provide in it and stores.

In [8]:
vector_store.add_documents(docs) #adds all in one collection named sample and generates a unique id for each doc

['fdda9883-14ba-4fe2-a5c5-c6d4eea14a78',
 '60846cee-e170-466d-a37c-23ccf0ddf8fd',
 '3b45efac-5ea8-47a0-bb4e-f2dacaacdab6',
 'fa2f17c9-d9bd-4484-a732-c582504f02dc',
 '0b5c29bc-8ff4-455d-a30b-076fbc0479e5']

In [11]:
#To check what's stored in your DB currently

vector_store.get(include= ['embeddings' , 'documents', 'metadatas'])

{'ids': ['fdda9883-14ba-4fe2-a5c5-c6d4eea14a78',
  '60846cee-e170-466d-a37c-23ccf0ddf8fd',
  '3b45efac-5ea8-47a0-bb4e-f2dacaacdab6',
  'fa2f17c9-d9bd-4484-a732-c582504f02dc',
  '0b5c29bc-8ff4-455d-a30b-076fbc0479e5'],
 'embeddings': array([[ 0.00994719,  0.0691433 , -0.05147115, ..., -0.03543334,
          0.01284806,  0.01248283],
        [ 0.00127745,  0.03129857, -0.02375379, ..., -0.00518358,
         -0.03280609,  0.02737712],
        [-0.10265926,  0.02650808,  0.02271502, ..., -0.0335975 ,
         -0.0798495 , -0.01507709],
        [ 0.02123396, -0.02468547, -0.04494365, ..., -0.10995812,
          0.00572562,  0.09915373],
        [ 0.01873973,  0.04382843, -0.04304251, ..., -0.07801621,
         -0.07840677, -0.0030419 ]], shape=(5, 384)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the mo

In [13]:
#retrieve related content upon querying

vector_store.similarity_search(query = "Who among them are bowler? ", k = 2) #k means top k vectors

[Document(id='fa2f17c9-d9bd-4484-a732-c582504f02dc', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(id='60846cee-e170-466d-a37c-23ccf0ddf8fd', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.")]

In [15]:
#similarity search with scores -> the lesser number the better (since it represents distance of vectors)

vector_store.similarity_search_with_score(query = "Who among them are bowler? ", k = 3)

[(Document(id='fa2f17c9-d9bd-4484-a732-c582504f02dc', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.9512463808059692),
 (Document(id='60846cee-e170-466d-a37c-23ccf0ddf8fd', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure."),
  1.1447954177856445),
 (Document(id='0b5c29bc-8ff4-455d-a30b-076fbc0479e5', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  1.1724748611450195)]

In [ ]:
#meta data based filtering

vector_store.similarity_search(
    query ="",
    filter= {"team": "Chennai Super Kings"} # Be case sensitive in CSK
)

[Document(id='3b45efac-5ea8-47a0-bb4e-f2dacaacdab6', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
 Document(id='0b5c29bc-8ff4-455d-a30b-076fbc0479e5', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.')]

In [22]:
#update an existing document with newer version

updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

#updating
vector_store.update_document(document_id="fdda9883-14ba-4fe2-a5c5-c6d4eea14a78", document=updated_doc1)

In [23]:
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['fdda9883-14ba-4fe2-a5c5-c6d4eea14a78',
  '60846cee-e170-466d-a37c-23ccf0ddf8fd',
  '3b45efac-5ea8-47a0-bb4e-f2dacaacdab6',
  'fa2f17c9-d9bd-4484-a732-c582504f02dc',
  '0b5c29bc-8ff4-455d-a30b-076fbc0479e5'],
 'embeddings': array([[-0.00233743,  0.05902078, -0.04774045, ..., -0.07264046,
          0.00276786, -0.0034409 ],
        [ 0.00127745,  0.03129857, -0.02375379, ..., -0.00518358,
         -0.03280609,  0.02737712],
        [-0.10265926,  0.02650808,  0.02271502, ..., -0.0335975 ,
         -0.0798495 , -0.01507709],
        [ 0.02123396, -0.02468547, -0.04494365, ..., -0.10995812,
          0.00572562,  0.09915373],
        [ 0.01873973,  0.04382843, -0.04304251, ..., -0.07801621,
         -0.07840677, -0.0030419 ]], shape=(5, 384)),
 'documents': ["Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple ce

In [25]:
#deleting an existing doc

vector_store.delete(ids = ["fdda9883-14ba-4fe2-a5c5-c6d4eea14a78"]) 

In [ ]:
vector_store.get(include = ['documents']) #virat deleted

{'ids': ['60846cee-e170-466d-a37c-23ccf0ddf8fd',
  '3b45efac-5ea8-47a0-bb4e-f2dacaacdab6',
  'fa2f17c9-d9bd-4484-a732-c582504f02dc',
  '0b5c29bc-8ff4-455d-a30b-076fbc0479e5'],
 'embeddings': None,
 'documents': ["Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
  'MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.',
  'Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.',
  'Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'],
 'uris': None,
 'included': ['documents'],
 'data': None,
 'metadatas': None}